This notebook creates customer-level features representing:

- Credit utilisation
- Bill balances
- Payment behaviour
- Repayment consistency
- Repayment delays
- Changes in account balances

All features are calculated using information recorded between April and
September 2005. The target records default in the following month, so no future
outcome information is used when constructing the predictors.

Import the libraries required for data manipulation, numerical calculations and
visualisation. Identify the project root so that files can be loaded using
reproducible relative paths.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.options.display.float_format = "{:,.2f}".format

In [ ]:
current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

print(f"Project root: {project_root}")

Load the output produced by the Day 2 notebook. Basic checks confirm that the
expected number of customers is present, customer identifiers remain unique and
the target contains only the documented binary values.

In [ ]:
data_path = (
    project_root
    / "data"
    / "processed"
    / "credit_default_cleaned.csv"
)

df = pd.read_csv(data_path)

assert len(df) == 30_000
assert df["customer_id"].is_unique
assert set(df["default_next_month"].unique()) == {0, 1}

print("Dataset shape:", df.shape)
df.head()

The dataset contains six months of repayment status, bill balance and payment
information. Creating lists for these columns makes it possible to calculate
features across the complete observation period without repeating individual
column names.

In [ ]:
repayment_columns = [
    "repayment_status_sep",
    "repayment_status_aug",
    "repayment_status_jul",
    "repayment_status_jun",
    "repayment_status_may",
    "repayment_status_apr",
]

bill_columns = [
    "bill_amount_sep",
    "bill_amount_aug",
    "bill_amount_jul",
    "bill_amount_jun",
    "bill_amount_may",
    "bill_amount_apr",
]

payment_columns = [
    "payment_amount_sep",
    "payment_amount_aug",
    "payment_amount_jul",
    "payment_amount_jun",
    "payment_amount_may",
    "payment_amount_apr",
]

In [ ]:
required_columns = {
    "customer_id",
    "credit_limit",
    "default_next_month",
    "months_with_recorded_delay",
    "maximum_recorded_delay",
    *repayment_columns,
    *bill_columns,
    *payment_columns,
}

missing_columns = required_columns.difference(df.columns)

assert not missing_columns, (
    f"Missing required columns: {sorted(missing_columns)}"
)

print("All required columns are present.")

Negative bill balances may represent refunds, overpayments or account credits.
They are retained in the original columns but treated as zero outstanding debt
when calculating utilisation.

Negative repayment-status codes are similarly retained in the original data but
treated as zero recorded delay when constructing numerical delay features.

In [ ]:
positive_bills = df[bill_columns].clip(lower=0)
recorded_delays = df[repayment_columns].clip(lower=0)

Credit utilisation compares the customer's outstanding balance with their
available credit limit

Higher utilisation may indicate greater dependence on available credit or
greater financial pressure.

Three utilisation measures are created:

- **Latest utilisation:** September balance divided by the credit limit
- **Average utilisation:** average positive balance across six months divided by the limit
- **Maximum utilisation:** highest positive balance divided by the limit

Utilisation may exceed 100% if a balance is greater than the assigned credit
limit, so values above 1 are not automatically treated as errors.

In [ ]:
df["latest_utilisation"] = (
    positive_bills["bill_amount_sep"]
    / df["credit_limit"]
)

df["average_utilisation"] = (
    positive_bills.mean(axis=1)
    / df["credit_limit"]
)

df["maximum_utilisation"] = (
    positive_bills.max(axis=1)
    / df["credit_limit"]
)

Average balances and payments summarise the customer's behaviour over the full
six-month observation period.

The change in bill balance is scaled by the credit limit, allowing balance
growth to be compared across customers with accounts of different sizes:

A positive result indicates that the balance increased, while a negative result
indicates that it decreased.

In [ ]:
df["average_positive_bill_amount"] = (
    positive_bills.mean(axis=1)
)

df["average_payment_amount"] = (
    df[payment_columns].mean(axis=1)
)

df["bill_change_to_limit"] = (
    positive_bills["bill_amount_sep"]
    - positive_bills["bill_amount_apr"]
) / df["credit_limit"]

Payment consistency measures how regularly the customer made a positive payment
during the six observed months.

The number of months with a payment is divided by six to produce a consistency
rate between 0 and 1. A value of 1 means that a positive payment was recorded
every month.

This measures payment regularity, not payment adequacy. A customer may make a
small payment every month without paying a substantial proportion of their
balance.

In [ ]:
df["months_with_payment"] = (
    df[payment_columns] > 0
).sum(axis=1)

df["payment_consistency_rate"] = (
    df["months_with_payment"] / len(payment_columns)
)

The total amount paid across six months is compared with the total positive bill
balance over the same period.

Because statement and payment dates may not align perfectly, this ratio is an
approximate behavioural indicator rather than an exact proportion of debt
repaid.

Customers with no positive bill balance receive a missing ratio because the
calculation has no meaningful denominator. A separate indicator records these
customers so that this information is not lost.

In [ ]:
total_positive_bills = positive_bills.sum(axis=1)
total_payments = df[payment_columns].sum(axis=1)

df["payment_to_bill_ratio"] = (
    total_payments
    / total_positive_bills.replace(0, np.nan)
)

df["no_positive_bill_history"] = (
    total_positive_bills == 0
).astype(int)

Average recorded delay summarises the severity of repayment problems across the
six-month observation period.

Financial values are often strongly right-skewed, with a small number of very
large observations. The transformation `log(1 + x)` reduces this skew while
retaining zero values.

Both the original and transformed features are retained so that their
distributions and modelling performance can be compared later.

In [ ]:
df["average_delay_months"] = (
    recorded_delays.mean(axis=1)
)

df["log_average_bill_amount"] = np.log1p(
    df["average_positive_bill_amount"]
)

df["log_average_payment_amount"] = np.log1p(
    df["average_payment_amount"]
)

df["log_payment_to_bill_ratio"] = np.log1p(
    df["payment_to_bill_ratio"]
)

The following checks confirm that:

- Utilisation and monetary features are non-negative
- Payment consistency lies between 0 and 1
- The number of payment months lies between 0 and 6
- No infinite values were introduced
- Missing values are limited to ratios with no valid bill denominator

In [ ]:
engineered_features = [
    "latest_utilisation",
    "average_utilisation",
    "maximum_utilisation",
    "average_positive_bill_amount",
    "average_payment_amount",
    "bill_change_to_limit",
    "months_with_payment",
    "payment_consistency_rate",
    "payment_to_bill_ratio",
    "no_positive_bill_history",
    "average_delay_months",
    "log_average_bill_amount",
    "log_average_payment_amount",
    "log_payment_to_bill_ratio",
]

In [ ]:
non_negative_features = [
    "latest_utilisation",
    "average_utilisation",
    "maximum_utilisation",
    "average_positive_bill_amount",
    "average_payment_amount",
    "months_with_payment",
    "payment_consistency_rate",
    "average_delay_months",
]

assert (df[non_negative_features] >= 0).all().all()

assert df["payment_consistency_rate"].between(
    0, 1
).all()

assert df["months_with_payment"].between(
    0, 6
).all()

assert not np.isinf(
    df[engineered_features]
).any().any()

print("Feature validation checks passed.")

In [ ]:
feature_missing_values = (
    df[engineered_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

feature_missing_values

Summary statistics are examined to understand the scale, spread, skewness and
extreme values of the new variables.

The 1st and 99th percentiles help distinguish generally unusual observations
from a small number of particularly extreme values. These results will inform
the preprocessing decisions used for modelling.

In [ ]:
feature_summary = (
    df[engineered_features]
    .describe(
        percentiles=[
            0.01,
            0.25,
            0.50,
            0.75,
            0.99,
        ]
    )
    .T
)

feature_summary

Average utilisation is divided into four approximately equal-sized customer
groups. Payment consistency is grouped according to the number of months in
which a positive payment was recorded.

These groups are created for descriptive analysis and visualisation. The
underlying continuous variables will remain available for modelling.

In [ ]:
df["average_utilisation_band"] = pd.qcut(
    df["average_utilisation"],
    q=4,
    labels=[
        "Lowest 25%",
        "Lower-middle 25%",
        "Upper-middle 25%",
        "Highest 25%",
    ],
)

df["payment_consistency_group"] = pd.cut(
    df["months_with_payment"],
    bins=[-1, 0, 3, 5, 6],
    labels=[
        "No recorded payments",
        "Payments in 1–3 months",
        "Payments in 4–5 months",
        "Payments every month",
    ],
)

In [ ]:
display(
    df["average_utilisation_band"]
    .value_counts(sort=False)
)

display(
    df["payment_consistency_group"]
    .value_counts(sort=False)
)

Default rates are calculated across utilisation and payment-consistency groups
to examine whether the engineered features separate customers into meaningfully
different risk segments.

These comparisons are descriptive. They do not prove that utilisation or
payment frequency directly causes default.

In [ ]:
def create_default_summary(data, column_name):
    summary = (
        data.groupby(
            column_name,
            observed=False
        )["default_next_month"]
        .agg(
            customers="size",
            defaults="sum",
            default_rate="mean",
        )
        .reset_index()
    )

    summary["default_rate_percentage"] = (
        summary["default_rate"] * 100
    )

    return summary

In [ ]:
utilisation_summary = create_default_summary(
    df,
    "average_utilisation_band"
)

payment_consistency_summary = create_default_summary(
    df,
    "payment_consistency_group"
)

display(utilisation_summary)
display(payment_consistency_summary)

The following charts compare default rates across average-utilisation quartiles
and payment-consistency groups.

A feature may be useful for risk assessment when it produces a clear and
reasonably consistent separation between groups. Formal predictive value will
be assessed during modelling rather than inferred from these charts alone.

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 5)
)

axes[0].bar(
    utilisation_summary[
        "average_utilisation_band"
    ].astype(str),
    utilisation_summary["default_rate"],
    color="#4C78A8",
)

axes[0].set_title(
    "Default Rate by Average Utilisation"
)
axes[0].set_xlabel("Average-utilisation quartile")
axes[0].set_ylabel("Default rate")

axes[1].bar(
    payment_consistency_summary[
        "payment_consistency_group"
    ].astype(str),
    payment_consistency_summary["default_rate"],
    color="#59A14F",
)

axes[1].set_title(
    "Default Rate by Payment Consistency"
)
axes[1].set_xlabel("Payment-consistency group")
axes[1].set_ylabel("Default rate")

for ax in axes:
    ax.yaxis.set_major_formatter(
        plt.FuncFormatter(
            lambda value, _: f"{value:.0%}"
        )
    )

    ax.tick_params(
        axis="x",
        labelrotation=20
    )

    for bar in ax.patches:
        height = bar.get_height()

        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height,
            f"{height:.1%}",
            ha="center",
            va="bottom",
        )

plt.tight_layout()

figure_path = (
    project_root
    / "outputs"
    / "figures"
    / "engineered_feature_default_rates.png"
)

plt.savefig(
    figure_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

Pearson correlations provide an initial indication of the direction and
strength of the linear relationship between each engineered feature and the
binary default outcome.

Correlation is not equivalent to causation or model importance. Non-linear
relationships and interactions may not be captured by this calculation.

In [ ]:
correlation_table = (
    df[
        engineered_features
        + ["default_next_month"]
    ]
    .corr()["default_next_month"]
    .drop("default_next_month")
    .sort_values(ascending=False)
    .to_frame("correlation_with_default")
)

correlation_table

Save the complete feature-engineered dataframe for modelling. Summary tables
are saved separately to support reporting and reproducibility.

The next notebook will split the data into training and test sets, preprocess
the predictors and build an interpretable logistic-regression baseline.

In [ ]:
featured_data_path = (
    project_root
    / "data"
    / "processed"
    / "credit_default_features.csv"
)

df.to_csv(
    featured_data_path,
    index=False
)

utilisation_summary.to_csv(
    project_root
    / "outputs"
    / "tables"
    / "utilisation_risk_summary.csv",
    index=False,
)

payment_consistency_summary.to_csv(
    project_root
    / "outputs"
    / "tables"
    / "payment_consistency_risk_summary.csv",
    index=False,
)

feature_summary.to_csv(
    project_root
    / "outputs"
    / "tables"
    / "engineered_feature_summary.csv"
)

correlation_table.to_csv(
    project_root
    / "outputs"
    / "tables"
    / "engineered_feature_correlations.csv"
)

print(f"Featured dataset saved to: {featured_data_path}")

Reload the saved dataset and confirm that its dimensions, customer identifiers
and target values match the in-memory dataframe. This ensures that the output
required for modelling was saved successfully.

In [ ]:
saved_df = pd.read_csv(featured_data_path)

assert saved_df.shape == df.shape
assert saved_df["customer_id"].is_unique
assert set(
    saved_df["default_next_month"].unique()
) == {0, 1}

print("Saved dataset successfully validated.")
print("Saved shape:", saved_df.shape)



The analysis created features representing credit utilisation, average account
balances, payment behaviour, repayment consistency and recorded delay severity.

Customers in the highest average-utilisation quartile had the highest
observed default rate at 29.96%, compared with 14.77% among customers
in the lower-middle utilisation quartile. The relationship was not entirely
linear, as the lowest-utilisation quartile had a default rate of 21.13%.

Customers with no recorded payments had a default rate of 37.99%, while
customers who made payments every month had the lowest default rate at
13.92%. This suggests that consistent payment activity is associated with
lower subsequent default risk.

Average recorded delay had the strongest positive correlation with
subsequent default (r = 0.376), followed by average utilisation
(r = 0.116) and latest utilisation** (r = 0.086). Payment consistency
had the strongest negative correlation (r = -0.163), followed by
log-transformed average payment amount (r = -0.153). These correlations
are descriptive and do not establish causation or predictive importance.

The payment-to-bill ratio was missing for 966 customers because they had no
positive bill balance during the observation period. These cases were retained
and identified using a separate binary indicator.

The feature-engineered dataset contains 30,000 unique customers and has been
saved for Day 4, when an interpretable logistic-regression baseline will be
developed and evaluated.